# Lecture 11: As fast as possible: minimum time and bang-bang control

**The challenge.** Industrial robots, disk-drive heads, cranes and satellites
re-pointing their telescopes all have the same goal: get there as fast as
the actuators allow. The cost is the time itself, $\int 1\,dt$, which says
nothing about the control. The control enters the dynamics linearly, so by
Lecture 4 it must sit at a bound and switch between bounds. The final time is
unknown. And for a system with several actuators, each switches on its own
schedule. Earlier lectures met these pieces one at a time. Here they come
together, along with some questions we can now answer: how many switches?
Where in the state space? Can the answer be given as a feedback law?

**After this lecture you can**

- set up a minimum-time problem with a free final time and its condition
  $H(t_f) = 0$;
- draw the switching curve of the double integrator and explain its
  feedback synthesis;
- state Feldbaum's theorem on the number of switches, and check it;
- solve a nonlinear, multi-actuator minimum-time problem (the COPS robot
  arm) from a costate distribution.

## 1. "Bang-bang"

The first minimum-time results came from relay control, the thermostats and
on-off servos of the 1940s, and from the phase-plane pictures engineers drew
of them. Donald Bushaw's 1952 Princeton thesis, written under Solomon
Lefschetz, proved that for a second-order system driven by a relay the
fastest way to the origin switches at most once, on a curve in the phase
plane that he constructed. In Moscow, Aleksandr Feldbaum proved in 1953 that
for a linear system of order $n$ whose eigenvalues are real, the
minimum-time control has at most $n - 1$ switches. At RAND, Richard Bellman,
Irving Glicksberg and Oliver Gross gave the problem its lasting name in a
1956 paper, "On the bang-bang control problem". Joseph LaSalle at RIAS in
Baltimore then showed in 1959, for linear systems, that whatever can be done
with a bounded control can be done as fast with a bang-bang one: the
*bang-bang principle*.

All of this came just before, or alongside, the minimum principle, which
explained it: the control is linear in $H$, so it sits at a bound decided by
the sign of a switching function.

## 2. The double integrator, and its switching curve

The prototype is $\ddot x = u$, $|u| \le u_{\max}$: a mass pushed by a
bounded force, to be brought to rest at the origin from $(x_0, v_0)$ as
fast as possible. $H = 1 + \lambda_x v + \lambda_v u$, so
$u = -u_{\max}\operatorname{sign}\lambda_v$, with $\dot\lambda_x = 0$ and
$\dot\lambda_v = -\lambda_x$. The switching function $S = \lambda_v$ is
*linear* in time, so it changes sign at most once: one switch at most, as
Bushaw found.

The final time is free, so it is an unknown of the boundary value problem,
and the extra condition is $H(t_f) = 0$ (Lecture 2). `derive` adds it:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from auto_optimal_control import examples
from auto_optimal_control.homotopy import Homotopy
from auto_optimal_control.pontryagin import derive

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

pmp = derive(examples.double_integrator_min_time())
b = pmp.boundary
print("unknowns:", b.unknowns)
print("H(t_f) = 0 residual:", b.hamiltonian)

Solve it from many starting points. Each solve is given only a box of
costates and a range of final times; everything else is found (Lecture 15
explains how).

In [ ]:
solver = Homotopy(examples.double_integrator_min_time())
starts = [(1.5, 0.0), (-1.0, 1.0), (0.5, 1.5), (-1.5, -0.5), (1.0, -1.2), (0.3, -1.6),
          (-0.4, 1.8), (1.8, 0.8)]
fig, ax = plt.subplots(figsize=(6.2, 5))
vv = np.linspace(-2.2, 2.2, 200)
ax.plot(-vv * np.abs(vv) / 2, vv, color="k", lw=1, ls="--", label=r"switching curve $x = -v|v|/2$")
for x0, v0 in starts:
    r = solver.solve([1.0, x0, v0], ([-3, -3], [3, 3]), final_time=(0.2, 6.0))
    tr = r.shooting.trajectory(r.solution.v, n_points=400)
    up = tr["u"][:, 0] > 0
    ax.plot(tr["z"][up, 0], tr["z"][up, 1], ".", color=ORANGE, ms=2)
    ax.plot(tr["z"][~up, 0], tr["z"][~up, 1], ".", color=BLUE, ms=2)
    ax.plot(x0, v0, "ko", ms=4)
ax.plot([], [], ".", color=ORANGE, label="u = +1")
ax.plot([], [], ".", color=BLUE, label="u = -1")
ax.set(xlabel="position x", ylabel="velocity v")
ax.legend(frameon=False, fontsize=8, loc="upper right", markerscale=5)
ax.grid(alpha=0.3)
fig.tight_layout()

Every extremal runs on a parabola of constant force until it meets the
dashed curve, then rides that curve into the origin with the opposite force.
The curve, $x = -v|v|/(2u_{\max})$, is the set of states from which a single
full-force push stops the mass exactly at the origin. It turns the open-loop
answer into a **feedback law**: push with $u = -u_{\max}$ above the curve and
$+u_{\max}$ below. That is Bushaw's synthesis, and for this one problem it is
the global solution (the value function of Lecture 5, explicitly).

## 3. How many switches? Feldbaum's theorem

For $\dot x = A x + B u$ with $|u| \le 1$, $S = B^\top\lambda$ and
$\dot\lambda = -A^\top\lambda$, so $S(t) = B^\top e^{-A^\top t}\lambda_0$. If
$A$ has real eigenvalues, $S$ is a sum of $n$ terms of the form
$t^k e^{\mu t}$, which can have at most $n - 1$ zeros: at most $n - 1$
switches. For the double integrator ($n = 2$) that is one. For a chain of
three integrators, a jerk-controlled axis ($\dddot x = u$), it is two. Check:

In [ ]:
import sympy as sp

from auto_optimal_control.problem import Box, OptimalControlProblem

t, x, v, a, u = sp.symbols("t x v a u", real=True)
jerk = OptimalControlProblem(
    name="triple_integrator_min_time", time=t, states=[x, v, a], controls=[u],
    dynamics=[v, a, u], running_cost=1, initial_state=[1, 0, 0],
    final_state={x: 0, v: 0, a: 0}, control_sets=[Box(u, -1, 1)],
)
rj = Homotopy(jerk).solve([], ([-3, -3, -3], [3, 3, 3]), final_time=(1.0, 8.0))
kj = rj.shooting.kernels
pr = kj.propagate(rj.solution.t0, rj.solution.z0, rj.solution.tf, rj.params)
print(f"t_f = {rj.solution.tf:.10f}, switches at {np.round(pr['switch_times'], 6)}")

Two switches, at $t_f/4$ and $3t_f/4$ by the symmetry of the problem:
$u = -1, +1, -1$. (For this start the closed form is $t_f = 2\cdot 2^{2/3}$,
about 3.1748.)

## 4. A nonlinear robot: the COPS arm

Real arms are not chains of integrators. The COPS test problem (Dolan, Moré
and Munson's collection of large optimization problems) models a robot arm
with three joints: a prismatic joint that extends the arm (length $\rho$),
and two rotations, $\theta$ about the vertical and $\varphi$ in elevation.
The inertias depend on the configuration, which couples the joints
nonlinearly:
$$
L\,\ddot\rho = u_\rho, \qquad I_\theta(\rho, \varphi)\,\ddot\theta = u_\theta, \qquad
I_\varphi(\rho)\,\ddot\varphi = u_\varphi, \qquad |u_i| \le 1,
$$
with $I_\varphi = ((L - \rho)^3 + \rho^3)/3$ and
$I_\theta = I_\varphi \sin^2\varphi$. Move from rest to rest, rotating by
$2\pi/3$ about the vertical, in minimum time.

In [ ]:
rho, th, ph, wr, wt, wp = sp.symbols("rho theta phi rho_d theta_d phi_d", real=True)
ur, ut, up = sp.symbols("u_rho u_theta u_phi", real=True)
Lsym = sp.Symbol("L", positive=True)
I_phi = ((Lsym - rho) ** 3 + rho**3) / 3
I_theta = I_phi * sp.sin(ph) ** 2
arm = OptimalControlProblem(
    name="robot_arm", time=t, states=[rho, th, ph, wr, wt, wp], controls=[ur, ut, up],
    params=[Lsym], dynamics=[wr, wt, wp, ur / Lsym, ut / I_theta, up / I_phi], running_cost=1,
    initial_state=[sp.Rational(9, 2), 0, sp.pi / 4, 0, 0, 0],
    final_state={rho: sp.Rational(9, 2), th: 2 * sp.pi / 3, ph: sp.pi / 4, wr: 0, wt: 0, wp: 0},
    control_sets=[Box(ur, -1, 1), Box(ut, -1, 1), Box(up, -1, 1)],
)
ra = Homotopy(arm).solve(
    [5.0], lambda rng: rng.uniform(-1, 1, 6) * 10 ** rng.uniform(-1, 1, 6), (5.0, 15.0),
    n_tries=300)
print(ra.message)
print(f"t_f = {ra.solution.tf:.8f}  (COPS reference 9.1409)")

In [ ]:
tr = ra.shooting.trajectory(ra.solution.v, n_points=1500)
fig, axes = plt.subplots(2, 3, figsize=(12, 5), sharex=True)
names = [r"$\rho$", r"$\theta$", r"$\varphi$"]
for j in range(3):
    axes[0, j].plot(tr["t"], tr["z"][:, j], color=BLUE, lw=2)
    axes[0, j].set_title(names[j], fontsize=11, loc="left")
    axes[1, j].plot(tr["t"], tr["u"][:, j], color=ORANGE, lw=2)
    axes[1, j].set_title(f"torque on {names[j]}", fontsize=10, loc="left")
    axes[1, j].set_xlabel("t")
    axes[1, j].axvline(ra.solution.tf / 2, color=GRAY, lw=0.8, ls=":")
for a_ in axes.ravel():
    a_.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# The COPS arm in space: a bar of length L slides through a pivot (rho is the length on the
# gripper side), turns about the vertical by theta and tilts from it by phi. Drawn at
# eight instants of the minimum-time motion, with the gripper's path.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3-D projection)

L_ARM = 5.0
rho_t, th_t, ph_t = tr["z"][:, 0], tr["z"][:, 1], tr["z"][:, 2]
direction = np.stack([np.sin(ph_t) * np.cos(th_t), np.sin(ph_t) * np.sin(th_t), np.cos(ph_t)], 1)
grip = rho_t[:, None] * direction
fig = plt.figure(figsize=(10, 4.6))
ax = fig.add_subplot(1, 2, 1, projection="3d")
picks = np.linspace(0, len(tr["t"]) - 1, 8).astype(int)
for n, i in enumerate(picks):
    alpha = 0.25 + 0.75 * n / (len(picks) - 1)
    back = -(L_ARM - rho_t[i]) * direction[i]
    ax.plot(*np.array([back, grip[i]]).T, color=BLUE, lw=2.5, alpha=alpha)
    ax.scatter(*grip[i], color=BLUE, s=18, alpha=alpha)
ax.plot(*grip.T, color=ORANGE, lw=1.5, label="gripper path")
ax.plot([0, 0], [0, 0], [-1.5, 0], color="k", lw=3)
ax.scatter(0, 0, 0, color="k", s=30)
ax.set(xlabel="x", ylabel="y", zlabel="z")
ax.view_init(elev=28, azim=-60)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title(r"rotate by $2\pi/3$ about the vertical, rest to rest", fontsize=10, loc="left")
right = fig.add_subplot(1, 2, 2)
for n, i in enumerate(picks):
    alpha = 0.25 + 0.75 * n / (len(picks) - 1)
    back = -(L_ARM - rho_t[i]) * direction[i]
    right.plot([back[0], grip[i, 0]], [back[1], grip[i, 1]], color=BLUE, lw=2.5, alpha=alpha)
right.plot(grip[:, 0], grip[:, 1], color=ORANGE, lw=1.5)
right.plot(0, 0, "ko", ms=6)
arc = np.linspace(0, 2 * np.pi / 3, 60)
right.plot(1.0 * np.cos(arc), 1.0 * np.sin(arc), color="k", lw=0.8)
right.text(0.75, 0.75, r"$\theta$: $0 \to 2\pi/3$", fontsize=9)
right.set_aspect("equal")
right.set(xlabel="x", ylabel="y")
right.set_title("from above: the gripper side pulls in mid-turn,\nwhich lowers the inertia",
                fontsize=10, loc="left")
right.grid(alpha=0.25)
fig.tight_layout()

*The same extremal in space. Every joint is bang-bang, and the motion is not the obvious one. The arm shortens its long side while it turns, lowering its moment of inertia, much as a skater pulls in their arms, then lengthens it again to arrive at rest.*

All three torques are bang-bang, each switching on its own schedule.
$u_\theta$, the base rotation, switches exactly at $t_f/2$: accelerate for
half the time, brake for the other half, as the symmetry of a rest-to-rest
rotation demands. The other two joints have to *move* to help. The arm
pulls in (smaller $\rho$, smaller inertia) and lowers its elevation during
the turn, then restores both, because a compact arm turns faster. Nothing in
the problem says so; it is what the minimum principle found.

**$H \equiv 0$ as a check.** The final time is free and the problem is
autonomous, so $H$ must vanish along the whole extremal:

In [ ]:
print(f"max |H(t)| along the arm's extremal: {np.abs(tr['H']).max():.1e}")

## Summary

- Minimum-time problems have $L = 1$, a free final time with the extra
  condition $H(t_f) = 0$ (hence $H \equiv 0$ when autonomous), and controls
  linear in $H$, so bang-bang.
- For the double integrator the switching curve $x = -v|v|/2u_{\max}$ gives
  a global feedback synthesis; in general the indirect method gives open-loop
  extremals, one start at a time.
- Feldbaum: a linear system of order $n$ with real eigenvalues needs at most
  $n - 1$ switches.
- Nonlinear, multi-actuator problems such as the COPS arm have one switching
  function per actuator; the package locates all their switches and finds the
  structure from a costate distribution alone.

## Exercises

1. Add damping to the double integrator, $\ddot x = u - c\,\dot x$. Does the
   switching curve survive? Plot it numerically by solving from many starts.
2. A harmonic oscillator $\ddot x = -x + u$ has complex eigenvalues. Solve
   minimum-time problems from farther and farther away and count the
   switches. What replaces Feldbaum's bound?
3. In the robot arm, fix $\rho$ and $\varphi$ (remove their dynamics and
   controls). How much slower is the rotation? What does that say about why
   the full solution moves all three joints?

## Further reading

- D. W. Bushaw, *Differential Equations with a Discontinuous Forcing Term*,
  PhD thesis, Princeton (1952).
- R. Bellman, I. Glicksberg and O. Gross, "On the 'bang-bang' control
  problem", *Quarterly of Applied Mathematics* 14, 1956.
- A. A. Feldbaum, *Optimal Control Systems* (Academic Press, 1965), on the
  $n - 1$ theorem.
- E. D. Dolan, J. J. Moré and T. S. Munson, *Benchmarking Optimization
  Software with COPS 3.0*, Argonne report ANL/MCS-TM-273 (2004), problem 8.